# Exercise: Text Embeddings, Similarity, and Retrieval Chatbot

This notebook helps students practice the techniques using SentenceTransformer, but it uses a **student services and academic advising** context instead of general AI examples.

After completing this exercise, students should be able to:

- Convert sentences/texts into embedding vectors.
- Compute semantic similarity between multiple sentences.
- Create a representative vector for a short document.
- Retrieve the most relevant document for a user query.
- Build a simple FAQ chatbot by finding the nearest stored question.

> Note: The example data uses English so the compact `all-MiniLM-L6-v2` model works more reliably. All instructions and response prompts in this exercise are written in English.


## 0. Environment Setup

Run the cell below first. If the machine does not have the `sentence-transformers` library installed, students need to install it before running this notebook.


In [23]:
import re

import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer


MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Load the embedding model; the first run may take a few minutes if the model must be downloaded.
model = SentenceTransformer(MODEL_NAME)

print("Model loaded:", MODEL_NAME)
print("Embedding dimension:", model.get_sentence_embedding_dimension())


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3516.19it/s]


Model loaded: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


/tmp/ipykernel_31602/3602855914.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


## 1. Exercise 1 - Convert a Sentence into a Vector

Complete the `encode_texts` function to encode a list of sentences into an embedding matrix.

Requirements:

1. Call the `embedding_model.encode` method.
2. Return the result as a NumPy array.
3. When `normalize=True`, vectors should be normalized so the dot product can be used as cosine similarity.
4. Run the testing cell and observe the vector shape and the first 10 values.


In [24]:
def encode_texts(texts, embedding_model, normalize=True):
    """Encode a list of texts into an embedding matrix.

    Parameters:
        texts: A list of text strings to encode.
        embedding_model: A SentenceTransformer model used to create embeddings.
        normalize: If True, normalize vectors so the dot product equals cosine similarity.

    Returns:
        A NumPy array containing one embedding vector for each input text.
    """
    # TODO 1: Call embedding_model.encode with convert_to_numpy=True.
    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy = True,
        # TODO 2: Pass normalize_embeddings=normalize to normalize vectors when needed.
        normalize_embeddings=normalize
    )
    return embeddings

In [25]:
campus_sentence = "Students can register for elective courses through the academic portal."

# After completing encode_texts, this cell will print the vector information for the sample sentence.
campus_vector = encode_texts([campus_sentence], model)[0]

print("Sentence:")
print(campus_sentence)
print("\nVector shape:", campus_vector.shape)
print("\nFirst 10 values:")
print(np.round(campus_vector[:10], 4))


Sentence:
Students can register for elective courses through the academic portal.

Vector shape: (384,)

First 10 values:
[ 0.0012 -0.0458 -0.0133  0.0349 -0.0105  0.0067 -0.0288 -0.0366 -0.0374
  0.0198]


## 2. Exercise 2 - Compare Sentence Similarity

In this section, students build a similarity matrix for sentences from different topics.

Requirements:

1. Complete `cosine_similarity_matrix`.
2. Complete `build_similarity_table` to display the matrix with `pandas.DataFrame`.
3. Observe the sentence pairs with high scores and explain why they are semantically similar.


In [26]:
def cosine_similarity_matrix(vectors):
    """Compute a pairwise cosine similarity matrix.

    Parameters:
        vectors: A 2D matrix containing normalized vectors.

    Returns:
        A square matrix where element (i, j) is the similarity between text i and text j.
    """
    # TODO: If vectors are normalized, cosine similarity is the dot product between vectors.
    return vectors @ vectors.T

def build_similarity_table(labels, similarity_matrix):
    """Create a readable similarity table from a score matrix.

    Parameters:
        labels: Row and column labels for the compared texts.
        similarity_matrix: A matrix of similarity scores.

    Returns:
        A DataFrame containing rounded similarity scores.
    """
    # TODO: Use pd.DataFrame, np.round, and labels for both index and columns.
    return pd.DataFrame(
        np.round(similarity_matrix, 3),
        index = labels,
        columns = labels
    )

service_sentences = [
    "Students can register for courses through the academic portal.",
    "Learners enroll in classes using the university website.",
    "The library allows students to borrow textbooks and research papers.",
    "Tuition payment deadlines are listed in the student portal.",
    "The cafeteria serves lunch from eleven o'clock.",
    "The basketball team practiced after school.",
]

sentence_labels = [f"S{i + 1}" for i in range(len(service_sentences))]

# Encode the sentences before computing similarity.
sentence_vectors = encode_texts(service_sentences, model)
sentence_similarity = cosine_similarity_matrix(sentence_vectors)

build_similarity_table(sentence_labels, sentence_similarity)


,S1,S2,S3,S4,S5,S6
S1,1.000,0.812,0.439,0.483,0.065,0.200
S2,0.812,1.000,0.462,0.443,0.114,0.244
S3,0.439,0.462,1.000,0.240,0.042,0.136
S4,0.483,0.443,0.240,1.000,0.132,0.130
S5,0.065,0.114,0.042,0.132,1.000,0.206
S6,0.200,0.244,0.136,0.130,0.206,1.000


In [27]:
def find_top_pairs(texts, similarity_matrix, top_k=5):
    """Find the sentence pairs with the highest semantic similarity.

    Parameters:
        texts: A list of original sentences.
        similarity_matrix: The similarity matrix between sentences.
        top_k: The number of sentence pairs to return.

    Returns:
        A DataFrame containing sentence pairs and their similarity scores.
    """
    rows = []

    # TODO 1: Iterate over pairs (i, j) with j > i to avoid duplicates and the main diagonal.
    for i in range(len(texts)):
        for j in range(i + 1, len(texts)):
            rows.append(
                {
                    # TODO 2: Each row should contain Text A, Text B, and Similarity.
                    "Sentence A": texts[i],
                    "Sentence B": texts[j],
                    "Similarity": similarity_matrix[i, j]
                }
            )
    # TODO 3: Sort by Similarity in descending order and return the first top_k rows.
    rows = sorted(rows, key=lambda item: item["Similarity"], reverse=True)
    return pd.DataFrame(rows[:top_k])
    

find_top_pairs(service_sentences, sentence_similarity, top_k=5)


,Sentence A,Sentence B,Similarity
0,Students can register for courses through the ...,Learners enroll in classes using the universit...,0.811725
1,Students can register for courses through the ...,Tuition payment deadlines are listed in the st...,0.482767
2,Learners enroll in classes using the universit...,The library allows students to borrow textbook...,0.461882
3,Learners enroll in classes using the universit...,Tuition payment deadlines are listed in the st...,0.442597
4,Students can register for courses through the ...,The library allows students to borrow textbook...,0.439479


## 3. Exercise 3 - Retrieve Short Documents

In the demo, several documents are compared with one another. In this exercise, students build a function that **finds the most relevant document for a query**.

Idea:

1. Split each document into smaller sentences.
2. Encode each sentence.
3. Average the sentence vectors to create a document vector.
4. Encode the user's query.
5. Compute similarity between the query and each document, then rank the results.


In [28]:
campus_documents = {
    "Course registration": (
        "Students use the academic portal to register for required and elective courses. "
        "Academic advisors can help students choose suitable classes for the semester."
    ),
    "Library services": (
        "The university library provides textbooks, journals, and quiet study rooms. "
        "Students can borrow books with their student ID card."
    ),
    "Tuition payment": (
        "Tuition fees can be paid online before the payment deadline. "
        "The finance office supports students who have questions about invoices."
    ),
    "Sports activities": (
        "Students can join sports clubs such as basketball, football, and badminton. "
        "Training sessions are announced by the student affairs office."
    ),
}


def split_sentences(text):
    """Split a text into short sentences using a simple regex rule.

    Parameters:
        text: The input text to split into sentences.

    Returns:
        A list of non-empty sentences.
    """
    # TODO: Use re.split with periods, question marks, and exclamation marks as sentence boundaries.
    pieces = re.split(r"(?<=[.!?])\s+", text.strip())
    return [piece for piece in pieces if piece]

def create_document_vector(document, embedding_model):
    """Create a representative document vector by averaging sentence embeddings.

    Parameters:
        document: The document text string.
        embedding_model: A SentenceTransformer model used to create embeddings.

    Returns:
        A normalized NumPy vector representing the document.
    """
    # TODO 1: Split the document into sentences with split_sentences.
    sentences_in_document = split_sentences(document)
    # TODO 2: Encode the sentences with encode_texts.
    sentence_vectors = encode_texts(sentences_in_document, embedding_model)
    # TODO 3: Average the sentence vectors.
    doc_vector = sentence_vectors.mean(axis=0)
    # TODO 4: Normalize the averaged vector before returning it.
    norm = np.linalg.norm(doc_vector)
    if norm == 0:
        return doc_vector
    return doc_vector / norm


def build_document_index(documents, embedding_model):
    """Build a vector index for a small document collection.

    Parameters:
        documents: A dictionary where keys are document names and values are document contents.
        embedding_model: A SentenceTransformer model used to create embeddings.

    Returns:
        A tuple containing the list of document names and the document vector matrix.
    """
    names = list(documents.keys())

    # TODO: Create one vector for each document, then combine them with np.vstack.
    document_vectors = np.vstack([
        create_document_vector(text, embedding_model)
        for text in documents.values()
    ])

    return names, document_vectors


document_names, document_matrix = build_document_index(campus_documents, model)
document_similarity = cosine_similarity_matrix(document_matrix)

build_similarity_table(document_names, document_similarity)


,Course registration,Library services,Tuition payment,Sports activities
Course registration,1.000,0.589,0.499,0.551
Library services,0.589,1.000,0.384,0.405
Tuition payment,0.499,0.384,1.000,0.306
Sports activities,0.551,0.405,0.306,1.000


In [30]:
def retrieve_documents(query, document_names, document_matrix, embedding_model, top_k=2):
    """Retrieve the documents that are most semantically similar to a user query.

    Parameters:
        query: The user's query.
        document_names: A list of document names.
        document_matrix: A matrix containing document vectors.
        embedding_model: A SentenceTransformer model used to encode the query.
        top_k: The number of nearest documents to return.

    Returns:
        A DataFrame containing document names and similarity scores in descending order.
    """
    # TODO 1: Encode the query into a vector.
    query_vector = encode_texts([query], embedding_model)[0]
    # TODO 2: Compute scores = document_matrix @ query_vector.
    scores = document_matrix @ query_vector
    # TODO 3: Sort indices by scores in descending order.
    sorted_indices = np.argsort(scores)[::-1]
    # TODO 4: Return a DataFrame with Document and Similarity columns.
    return pd.DataFrame({
        "Document": [document_names[i] for i in sorted_indices],
        "Similarity": [scores[i] for i in sorted_indices]
    })

sample_queries = [
    "Where can I borrow research books?",
    "How do I pay my tuition invoice?",
    "Can I join a basketball practice?",
]

for query in sample_queries:
    print("Query:", query)
    display(retrieve_documents(query, document_names, document_matrix, model, top_k=2))


Query: Where can I borrow research books?


,Document,Similarity
0,Library services,0.579168
1,Course registration,0.300855
2,Tuition payment,0.225158
3,Sports activities,0.130717


Query: How do I pay my tuition invoice?


,Document,Similarity
0,Tuition payment,0.740385
1,Course registration,0.312170
2,Library services,0.247939
3,Sports activities,0.209940


Query: Can I join a basketball practice?


,Document,Similarity
0,Sports activities,0.490241
1,Course registration,0.195312
2,Library services,0.128726
3,Tuition payment,0.118221


## 4. Exercise 4 - FAQ Chatbot for Student Services

This section extends the retrieval idea into a small FAQ chatbot.

Requirements:

1. Use the `faq_pairs` list as the initial data.
2. Add at least **3 new question-answer pairs** written by students.
3. Complete the function that builds the question index.
4. Complete the function that answers a question by retrieving the nearest stored question.
5. If the similarity is lower than `threshold`, the chatbot must respond that it is not confident enough.


In [31]:
faq_pairs = [
    {
        "question": "How can I register for a course?",
        "answer": "You can register for a course through the academic portal during the registration period.",
    },
    {
        "question": "Where can I borrow textbooks?",
        "answer": "You can borrow textbooks from the university library with your student ID card.",
    },
    {
        "question": "How do I pay tuition fees?",
        "answer": "Tuition fees can be paid online or at the finance office before the deadline.",
    },
    {
        "question": "Who can help me choose classes?",
        "answer": "Your academic advisor can help you choose classes that match your study plan.",
    },
    {
        "question": "How can I join a sports club?",
        "answer": "You can contact the student affairs office or register during club introduction week.",
    },
]

# TODO: Add at least 3 new question-answer pairs to the list below.
faq_pairs.extend([
    {
         "question": "How do I know where are all school's department?",
         "answer": "Right in the main lobby of the school, there is a map that specifies each area of ​​the school's departments. You can go there to check the room you want to find.",
    },
    {
         "question": "Where can I park my motorbike?",
         "answer": "You can park your motorbike in the parking lot behind the school with a student parking area",
    },
    {
         "question": "Will the school help me get a job?",
         "answer": "The school has a policy to support students in getting jobs after graduation",
    },
])

pd.DataFrame(faq_pairs)


,question,answer
0,How can I register for a course?,You can register for a course through the acad...
1,Where can I borrow textbooks?,You can borrow textbooks from the university l...
2,How do I pay tuition fees?,Tuition fees can be paid online or at the fina...
3,Who can help me choose classes?,Your academic advisor can help you choose clas...
4,How can I join a sports club?,You can contact the student affairs office or ...
5,How do I know where are all school's department?,"Right in the main lobby of the school, there i..."
6,Where can I park my motorbike?,You can park your motorbike in the parking lot...
7,Will the school help me get a job?,The school has a policy to support students in...


In [32]:
def build_faq_index(pairs, embedding_model):
    """Create an embedding matrix for all FAQ questions.

    Parameters:
        pairs: A list of dictionaries with 'question' and 'answer' keys.
        embedding_model: A SentenceTransformer model used to encode questions.

    Returns:
        An embedding matrix for the FAQ questions.
    """
    # TODO: Extract the question list from pairs and encode it with encode_texts.
    questions = [pair["question"] for pair in pairs]
    faq_embeddings = encode_texts(questions, embedding_model)
    return faq_embeddings

def answer_faq(user_question, pairs, faq_embeddings, embedding_model, threshold=0.70, top_k=3):
    """Answer a question by finding the nearest FAQ question.

    Parameters:
        user_question: The question entered by the user.
        pairs: A FAQ list containing questions and answers.
        faq_embeddings: A precomputed embedding matrix for the FAQ questions.
        embedding_model: A SentenceTransformer model used to encode the user question.
        threshold: The minimum similarity score required for a confident answer.
        top_k: The number of nearest questions to keep for inspection.

    Returns:
        A dictionary containing the answer, matched question, similarity score, and top matches.
    """
    # TODO 1: Encode user_question into a vector.
    user_embedding = encode_texts([user_question], embedding_model)[0]
    # TODO 2: Compute similarity between user_question and all faq_embeddings.
    scores = faq_embeddings @ user_embedding
    # TODO 3: Find the question with the highest score.
    ranked_indices = np.argsort(scores)[::-1]
    best_index = int(ranked_indices[0])
    best_score = float(scores[best_index])
    top_matches = [
        {
            "question": pairs[int(index)]["question"],
            "answer": pairs[int(index)]["answer"],
            "similarity": float(scores[int(index)]),
        }
        for index in ranked_indices[:top_k]
    ]
    # TODO 4: If the highest score < threshold, return a not-confident response.
    if best_score < threshold:
        answer = "I am not confident enough to answer this question from the training data."
    else:
        answer = pairs[best_index]["answer"]
    # TODO 5: Return a dictionary with user_question, answer, matched_question, similarity, and top_matches.
    return {
        "user_question": user_question,
        "answer": answer,
        "matched_question": pairs[best_index]["question"],
        "similarity": best_score,
        "top_matches": top_matches,
    }

faq_embeddings = build_faq_index(faq_pairs, model)

print("Number of FAQ questions:", len(faq_pairs))
print("FAQ embedding matrix shape:", faq_embeddings.shape)


Number of FAQ questions: 8
FAQ embedding matrix shape: (8, 384)


In [33]:
test_questions = [
    "How can I enroll in a class?",
    "Where can I find journal articles?",
    "When is lunch served in the cafeteria?",
    "Can the advisor choose all my courses for me?",
]

for question in test_questions:
    result = answer_faq(question, faq_pairs, faq_embeddings, model, threshold=0.70, top_k=3)

    print("=" * 90)
    print("User question:", result["user_question"])
    print("Best matched FAQ question:", result["matched_question"])
    print("Similarity:", round(result["similarity"], 3))
    print("Answer:", result["answer"])


User question: How can I enroll in a class?
Best matched FAQ question: How can I register for a course?
Similarity: 0.732
Answer: You can register for a course through the academic portal during the registration period.
User question: Where can I find journal articles?
Best matched FAQ question: Where can I borrow textbooks?
Similarity: 0.284
Answer: I am not confident enough to answer this question from the training data.
User question: When is lunch served in the cafeteria?
Best matched FAQ question: How do I know where are all school's department?
Similarity: 0.248
Answer: I am not confident enough to answer this question from the training data.
User question: Can the advisor choose all my courses for me?
Best matched FAQ question: Who can help me choose classes?
Similarity: 0.483
Answer: I am not confident enough to answer this question from the training data.


### Inspect the Nearest Questions

After completing the chatbot, run the cell below to view the 3 FAQ questions nearest to a query. This table helps students explain why the chatbot selected its answer.


In [34]:
question_to_inspect = "I need help paying my tuition invoice."
result = answer_faq(question_to_inspect, faq_pairs, faq_embeddings, model, threshold=0.7, top_k=3)

pd.DataFrame(result["top_matches"])


,question,answer,similarity
0,How do I pay tuition fees?,Tuition fees can be paid online or at the fina...,0.698482
1,How can I register for a course?,You can register for a course through the acad...,0.345208
2,Will the school help me get a job?,The school has a policy to support students in...,0.336423


## 5. Submission Section

Students should answer the following questions briefly inside the notebook:

1. Which sentence pair in Exercise 2 has the highest similarity? Why?
2. For each query in Exercise 3, which document is retrieved first? Is the result reasonable?
3. When the FAQ chatbot `threshold` is increased from `0.50` to `0.70`, which answers change?
4. Add a question that makes the chatbot answer incorrectly or without confidence. Explain the reason.
5. If you wanted better performance for Vietnamese data, what model or data changes would you suggest?


### Student Answers

1. The pair with the highest similarity is S1–S2 with a score of 0.812:

S1: "Students can register for courses through the academic portal."
S2: "Learners enroll in classes using the university website."

The two sentences use completely different words (register/enroll, courses/classes, portal/website) but the model recognizes that they have the same meaning because it is trained on large paraphrase data - this is the strength of semantic embedding compared to TF-IDF.

2.  The results are reasonable because the queries use words that are synonymous with the document content (borrow/library, invoice/tuition, basketball/sports).

    Query: Where can I borrow research books?
    Result: Library services	0.5791683

    Query: How do I pay my tuition invoice?
    Result: Tuition payment		0.7403854

    Query: Can I join a basketball practice?
    Result: Sports activities	0.49024132

3. The FAQ does not change when the threshold is increased to 0.70 because all FAQ answers have similarity > 0.70 with the corresponding query, so no questions are filtered out. The question students asked was not far from the topic of the FAQ

4. With questions like "Can I get a refund if I drop a course after the deadline?"

This query is between two domains (Tuition payment and Course registration) but asks about the refund policy — something that is not in any document. The model will return the closest document according to the vector, but the similarity is low and the answer is not really correct. This is a typical out-of-scope query.

5. Regarding the model, replace all-MiniLM with:
    - intfloat/multilingual-e5-base or BAAI/bge-m3 — good Vietnamese support, encodes both query and document in the same space
    - keepitreal/vietnamese-sbert — fine-tune on Vietnamese data
